# prometheus_fastapi_instrumentator smoke test

`prometheus_fastapi_instrumentator` adds request metrics and a Prometheus `/metrics` endpoint to a FastAPI app. Pure Python. Every request below goes through Starlette's in-process test client: no socket is opened and no network is needed.

## Example 1 — the installed version, and an instrumented app serving `/metrics`

In [ ]:
import importlib.metadata
import prometheus_fastapi_instrumentator
from fastapi import FastAPI
from fastapi.testclient import TestClient
from prometheus_client import CollectorRegistry
from prometheus_fastapi_instrumentator import Instrumentator

assert prometheus_fastapi_instrumentator.__version__ == importlib.metadata.version("prometheus_fastapi_instrumentator")
app = FastAPI()
@app.get("/ping")
def ping():
    return {"pong": True}
# A private registry per example: the default one is process-wide and would see the other cells' metrics.
Instrumentator(registry=CollectorRegistry()).instrument(app).expose(app)
client = TestClient(app)
assert client.get("/ping").json() == {"pong": True}
body = client.get("/metrics").text
assert 'http_requests_total{handler="/ping",method="GET",status="2xx"} 1.0' in body, body[:800]
print(f"prometheus_fastapi_instrumentator {prometheus_fastapi_instrumentator.__version__} -> /metrics has", len(body.splitlines()), "lines")

## Example 2 — routes added with `include_router` keep their own label

In [ ]:
from fastapi import APIRouter, FastAPI
from fastapi.testclient import TestClient
from prometheus_client import CollectorRegistry
from prometheus_fastapi_instrumentator import Instrumentator

router = APIRouter()
@router.get("/v1/models")
def models():
    return {"data": []}
app = FastAPI()
app.include_router(router)
registry = CollectorRegistry()
Instrumentator(registry=registry).instrument(app).expose(app)
client = TestClient(app)
assert client.get("/v1/models").status_code == 200
labelled = registry.get_sample_value("http_requests_total", {"handler": "/v1/models", "method": "GET", "status": "2xx"})
# Recent releases expand the router's routes themselves (`routing._effective_routes`); a `handler="none"` here would be the defect.
assert labelled == 1.0, client.get("/metrics").text[:800]
print('http_requests_total{handler="/v1/models"} =', labelled)

## Example 3 — an extra metric from a hand-written function, and status codes left ungrouped

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from prometheus_client import CollectorRegistry, Counter
from prometheus_fastapi_instrumentator import Instrumentator, metrics

app = FastAPI()
@app.get("/items/{item_id}")
def item(item_id: int):
    return {"item": item_id}
registry = CollectorRegistry()
by_status = Counter("demo_responses_total", "Responses by exact status.", labelnames=("handler", "status"), registry=registry)
def count_by_status(info: metrics.Info) -> None:
    by_status.labels(info.modified_handler, info.modified_status).inc()
instrumentator = Instrumentator(should_group_status_codes=False, registry=registry)
instrumentator.add(count_by_status).add(metrics.requests(metric_name="demo_requests_total", registry=registry))
instrumentator.instrument(app)
client = TestClient(app)
assert client.get("/items/7").status_code == 200 and client.get("/items/seven").status_code == 422
handler = "/items/{item_id}"
ok = registry.get_sample_value("demo_responses_total", {"handler": handler, "status": "200"})
refused = registry.get_sample_value("demo_responses_total", {"handler": handler, "status": "422"})
built_in = registry.get_sample_value("demo_requests_total", {"handler": handler, "method": "GET", "status": "422"})
assert (ok, refused, built_in) == (1.0, 1.0, 1.0), (ok, refused, built_in)
print("one 200 and one 422 counted on the templated route:", handler)

## Example 4 — excluded handlers are not counted; latency is recorded for the rest

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from prometheus_client import CollectorRegistry
from prometheus_fastapi_instrumentator import Instrumentator

app = FastAPI()
@app.get("/health")
def health():
    return {}
@app.get("/work")
def work():
    return {"done": True}
registry = CollectorRegistry()
Instrumentator(excluded_handlers=["/health"], registry=registry).instrument(app)
client = TestClient(app)
for path in ("/health", "/health", "/work", "/work", "/work"):
    assert client.get(path).status_code == 200
counted = registry.get_sample_value("http_requests_total", {"handler": "/work", "method": "GET", "status": "2xx"})
skipped = registry.get_sample_value("http_requests_total", {"handler": "/health", "method": "GET", "status": "2xx"})
timed = registry.get_sample_value("http_request_duration_seconds_count", {"handler": "/work", "method": "GET"})
assert (counted, skipped, timed) == (3.0, None, 3.0), (counted, skipped, timed)
print("/work counted 3 times with 3 latency samples; /health not counted")